# F02 ? Native CUDA RMSNorm correctness

Run this notebook from top to bottom in a T4 session. It retains F01 setup, fresh vector-smoke compilation and metadata export, then compiles the dispatcher-registered RMSNorm operator and runs its correctness suite.

This is correctness qualification, not a performance benchmark. Source changes require a new Python process; each test command below uses one. Export the archive even when a check fails. No native RMSNorm result is claimed until this notebook is executed.

## 1. Pin the source

A commit SHA identifies the exact source being tested. Copy the full 40-character commit from the feature PR, rather than testing a moving branch. A fresh directory prevents overwriting a previous attempt. This public checkout needs no GitHub token.

In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import venv

REVISION = "PASTE_FULL_COMMIT_SHA"
assert re.fullmatch(r"[0-9a-fA-F]{40}", REVISION), "Set REVISION to the PR's full commit SHA"
base = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
assert base.is_dir(), "Use a Colab or Kaggle Linux notebook"
workspace = Path(tempfile.mkdtemp(prefix="aegis-f02-", dir=base))
repo = workspace / "repo"
artifacts = workspace / "artifacts"
artifacts.mkdir()

def run(args, *, cwd=None, env=None, log_name="setup-and-tests.log"):
    completed = subprocess.run(args, cwd=cwd, env=env or globals().get("project_env"), text=True, capture_output=True)
    with (artifacts / log_name).open("a") as log:
        log.write("\n$ " + " ".join(map(str, args)) + "\n")
        log.write(completed.stdout + completed.stderr)
    print((completed.stdout + completed.stderr)[-4000:])
    completed.check_returncode()
    return completed

run(["git", "init", str(repo)])
run(["git", "remote", "add", "origin", "https://github.com/MutugiD/Aegis-Norm.git"], cwd=repo)
run(["git", "fetch", "--depth=1", "origin", REVISION], cwd=repo)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo)
actual = run(["git", "rev-parse", "HEAD"], cwd=repo).stdout.strip()
assert actual.lower() == REVISION.lower()
(artifacts / "commit.txt").write_text(actual + "\n")
print("Workspace:", workspace)

## 2. Inspect the notebook host

The **driver** lets the OS communicate with the GPU. The **CUDA toolkit** supplies `nvcc`, which compiles CUDA source on the VM's CPU. The **PyTorch wheel** supplies its own CUDA runtime dependencies; installing it does not install a complete compiler toolchain.

C2 uses PyTorch 2.14.0/cu126 and toolkit 12.6. This remains a qualification candidate. This cell selects toolkit 12.6 explicitly and installs its versioned toolkit package alongside the provider toolkit if absent. It does not request a driver package. This requires an Ubuntu NVIDIA package repository and root/sudo access; failures remain in the exported log. Review actual driver/toolkit versions after installation.

In [ ]:
run(["nvidia-smi"])

# Select toolkit 12.6 explicitly, alongside any provider-installed toolkit.
cuda_home = Path("/usr/local/cuda-12.6")
nvcc = cuda_home / "bin" / "nvcc"
if not nvcc.is_file():
    prefix = [] if os.geteuid() == 0 else ["sudo"]
    run(prefix + ["apt-get", "update"])
    run(prefix + ["apt-get", "install", "-y", "cuda-toolkit-12-6"])

assert nvcc.is_file(), "CUDA toolkit 12.6 installation did not provide nvcc"
os.environ["CUDA_HOME"] = str(cuda_home)
os.environ["CUDACXX"] = str(nvcc)
os.environ["PATH"] = str(cuda_home / "bin") + os.pathsep + os.environ["PATH"]
if "project_env" in globals():
    project_env["CUDA_HOME"] = str(cuda_home)
    project_env["CUDACXX"] = str(nvcc)
    project_env["PATH"] = str(cuda_home / "bin") + os.pathsep + project_env["PATH"]

toolkit = run([str(nvcc), "--version"]).stdout
assert re.search(r"release 12\.6\b", toolkit), "Expected the selected CUDA 12.6 compiler"
run(["c++", "--version"])

## 3. Install in an isolated notebook environment

Ubuntu can require a version-specific venv package for pip bootstrapping. This cell installs pythonX.Y-venv for the running interpreter before creating the environment through the logging helper. Re-running preserves the partial environment directory. A Python virtual environment keeps project dependencies separate from provider-installed packages. Its CPU compiler and GPU are still those of the notebook VM. We explicitly choose the cu126 wheel, then install the package and its build/test dependencies. The package wheel includes native source files; the next step compiles them for this session.

This can take several minutes and requires disk space for PyTorch and its dependencies. Model weights are not downloaded in this notebook.

In [ ]:
# Install venv/ensurepip support for the notebook's exact Python version.
prefix = [] if os.geteuid() == 0 else ["sudo"]
venv_package = f"python{sys.version_info.major}.{sys.version_info.minor}-venv"
run(prefix + ["apt-get", "update"])
run(prefix + ["apt-get", "install", "-y", venv_package])

environment = workspace / "venv"
# Reuse the partially created directory; do not delete the workspace.
run([sys.executable, "-m", "venv", str(environment)])
python = str(environment / "bin" / "python")
project_env = os.environ.copy()
project_env["PATH"] = str(environment / "bin") + os.pathsep + project_env["PATH"]

run([python, "-m", "pip", "--version"])
run([python, "-m", "pip", "install", "torch==2.14.0",
     "--index-url", "https://download.pytorch.org/whl/cu126"])
run([python, "-m", "pip", "install", "-r", str(repo / "requirements-foundation.txt")])
run([python, "-m", "pip", "install", "--no-deps", str(repo)])
run([python, "-m", "pip", "check"])
run([python, "-m", "pip_audit", "--format", "json",
     "--output", str(artifacts / "installed-audit.json")])
run([python, "-m", "pip_audit", "-r", str(repo / "requirements-foundation.txt"),
     "--no-deps", "--disable-pip", "--format", "json",
     "--output", str(artifacts / "direct-audit.json")])

## 4. Preflight and explicit native build

Preflight records the actual GPU, available memory, Python/PyTorch versions, compiler, toolkit and driver. `ready_for_build` means the build may be attempted; it is not a correctness result.

The smoke command invokes the C++/CUDA extension builder. Ninja coordinates compilation, `MAX_JOBS=2` limits host build parallelism, and architecture `7.5` targets T4. The binding receives a PyTorch tensor and passes its existing GPU pointer to the kernel; it does not copy the tensor through CPU memory.

The CUDA launcher uses PyTorch's current stream. Returning to Python does not mean the GPU is finished. The checks synchronize before comparing values, and exercise a non-default stream. Each standalone smoke uses a new build directory by default, then the tests reuse it. Compiler output is saved to `build.log`; this cell may be quiet while compilation runs.

In [ ]:
run([python, "-m", "aegis_norm.preflight", "--output",
     str(artifacts / "preflight.json"), "--require-t4"], cwd=repo)
run([python, "-m", "aegis_norm.smoke", "--output-root", str(artifacts)], cwd=repo)
# Reuse this fresh build for the following tests, avoiding a second compilation.
latest = max(artifacts.glob("*/result.json"), key=lambda p: p.stat().st_mtime_ns)
smoke_result = json.loads(latest.read_text())
assert smoke_result["status"] == "passed", "Inspect the smoke run before continuing"
project_env["TORCH_EXTENSIONS_DIR"] = smoke_result["build_directory"]


## 5. Run reference and native binding tests

The **reference** calculates RMSNorm with ordinary PyTorch operations. When its input tensor is on CUDA, those operations run on the T4. It accumulates in FP32 and casts normalized values before multiplying by the weight, preserving the specified rounding boundary.

`backend='cuda'` requires explicit native loading in each process: the vector smoke kernel must never masquerade as native RMSNorm. `explain_dispatch` exposes that distinction. GPU tests require explicit opt-in so a CPU CI job cannot silently claim to have qualified CUDA.

In [ ]:
test_env = project_env.copy()
test_env["AEGIS_RUN_GPU"] = "1"
run([python, "-m", "pytest", str(repo / "tests/test_reference.py"),
     str(repo / "tests/test_preflight.py"), str(repo / "tests/test_gpu_smoke.py"),
     "-v", "--junitxml=" + str(artifacts / "tests.xml")], cwd=workspace, env=test_env)

## 6. Native RMSNorm reduction and validation

Each block owns one token row. Its 256 threads accumulate squares in FP32; eight warps reduce to eight partial sums. The first warp zero-fills unused lanes, combines the partials and broadcasts the inverse RMS through shared memory. Each thread then scales its elements and applies the dtype conversion before multiplying by gamma.

The C++ dispatcher validates native inputs and rejects active gradients. Auto mode uses the reference for supported fallback cases. The fixture calls `load_native()` in the pytest process; a previous Python process cannot retain that registration for it.

`-s` preserves the native compiler output in the dedicated log even on passing tests. The suite does not loosen tolerances after failures. Full sanitizer and independent reproduction evidence remain separate gates.

In [ ]:
native_env = project_env.copy()
native_env["AEGIS_RUN_GPU"] = "1"
run([python, "-m", "pytest", str(repo / "tests/test_native_rmsnorm.py"),
     "-v", "-s", "--junitxml=" + str(artifacts / "rmsnorm-tests.xml")],
    cwd=workspace, env=native_env, log_name="native-build-and-tests.log")

## 7. Export evidence, including failures

Run this cell even if an earlier cell failed. The archive contains the commit, setup/test log, environment report and any smoke results. A successful smoke run additionally contains compiler output, case results, resolved package versions and SHA-256 file hashes.

Resolved versions are an environment snapshot, not proof that a different notebook will reproduce the run. Maintain the selected wheel index and native toolchain as recorded. F01 qualification remains pending until the run is reviewed. Full model/benchmark recovery workflows follow in F06.

In [ ]:
# Export partial evidence too, but identify omitted steps explicitly.
required = ["rmsnorm-tests.xml", "native-build-and-tests.log", "commit.txt", "preflight.json", "tests.xml",
            "installed-audit.json", "direct-audit.json"]
missing = [name for name in required if not (artifacts / name).is_file()]
smoke_records = sorted(artifacts.glob("*/result.json"))
if not smoke_records:
    missing.append("standalone smoke run (rerun the preflight/build cell)")
collection = {"missing": missing, "qualification": "requires_review"}
(artifacts / "collection-status.json").write_text(json.dumps(collection, indent=2) + "\n")
print("Missing evidence:", missing or "none detected; results still require review")
import hashlib
file_hashes = {p.relative_to(artifacts).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
               for p in sorted(artifacts.rglob("*")) if p.is_file() and p.name != "artifact-sha256.json"}
(artifacts / "artifact-sha256.json").write_text(json.dumps(file_hashes, indent=2) + "\n")
archive = shutil.make_archive(str(workspace / "aegis-f02-evidence"), "zip", artifacts)
print("Evidence archive:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(archive))
    print("On Kaggle, also download the archive from the notebook output files.")